In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join(r'C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src'))
container_file_path = '/credentials/robust-raster-cefaedc5282c.json'  # Path inside the container

if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
# Set up Dask cluster using Docker Python SDK
import docker

# Initialize Docker client
docker_client = docker.from_env()

# Create a Docker network
network_name = "dask-network"

try:
    docker_client.networks.get(network_name)
except docker.errors.NotFound:
    docker_client.networks.create(network_name, driver="bridge")

# Define paths for volume mounting
host_file_path = r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json'  # Host machine path
container_file_path = '/credentials/robust-raster-cefaedc5282c.json'  # Path inside the container

host_nc_path = r'C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\experiment_code\saved_on_disk.nc'
container_nc_path = '/data/saved_on_disk.nc'

# Volume mapping
volumes = {
    host_file_path: {'bind': container_file_path, 'mode': 'ro'},  # 'ro' means read-only
    host_nc_path: {'bind': container_nc_path, 'mode': 'ro'},
}

scheduler = docker_client.containers.run(
    "adrianomdocker/rrtest",
    command="dask-scheduler --preload dask_memusage --memusage-csv /tmp/memusage.csv",
    name="dask-scheduler",
    network=network_name,
    detach=True,
    ports={'8786/tcp': 8786, '8787/tcp': 8787},
)

print(f"Dask Scheduler started with ID {scheduler.id}")

# Run Dask Workers
num_workers = 2  # Specify the number of workers you want
n_threads = 1
container_mem_limit = "8GB"
dask_mem_limit = "8GB"

worker_ids = []
for i in range(num_workers):
    worker = docker_client.containers.run(
        "adrianomdocker/rrtest",
        command=f"dask-worker dask-scheduler:8786 --nthreads {n_threads} --memory-limit {dask_mem_limit}",
        name=f"dask-worker-{i+1}",
        network=network_name,
        detach=True,
        mem_limit=container_mem_limit,
        volumes=volumes
)
    worker_ids.append(worker.id)
    print(f"Dask Worker {i+1} started with ID {worker.id}")

# Connect to the Dask Scheduler
#dask_client = Client('tcp://localhost:8786')

print("Connected to Dask Scheduler")

# Monitor the Dask Dashboard
print("Dask dashboard available at http://localhost:8787")

# The cluster is now set up and can be used within the script or interactively.

In [ ]:
from dask.distributed import Client
dask_client = Client('tcp://localhost:8786')

In [ ]:
from dask_plugins import EEPlugin

ee_plugin = EEPlugin(container_file_path)
dask_client.register_plugin(ee_plugin)

In [ ]:
# Earth Engine xarray
import sys
import os
import ee
import json

json_key = r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json'
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)

ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')

def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)

from input_driver import EarthEngineData

CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")

chunk_size  = {
            'time': 48,
            'X': 512,
            'Y': 256
}

parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'bands': ['SR_B4', 'SR_B5'],
            'start_date': '2020-12-29',
            'end_date': '2020-12-31',
            'geometry': CALIFORNIA.geometry(),
            'crs': 'EPSG:3310',
            'scale': 100,
            'chunks': chunk_size,
            'map_function': prep_sr_l8
        }

ds = EarthEngineData(parameters).dataset

In [ ]:
print(ds)

In [ ]:
import xarray as xr
import pandas as pd
import sys, os

def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

from udf_tuner import UserDefinedFunction

user_defined_func = UserDefinedFunction()

# Apply the user-defined function
# If I load the dataset as a dask delayed, I don't need a template!
result = user_defined_func.apply_user_function(ds, process_chunk_as_pandas)